# Notebook 04 – Scenario Analysis

In this notebook, I answer one question:

**Given the manufacturing companies CSU already works with, where would investing effort help students the most?**

I do this by:
- estimating which jobs undergrads could realistically fill
- comparing that to CSU’s ability to place students
- testing a few simple “what if” situations

## 1) Load processed dataset & quick checks

Here I load the cleaned and modeled dataset from the previous notebooks.
This dataset already includes:
- role family
- pipeline level
- accessibility score

So I can focus on comparisons instead of cleaning.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA_PATH = Path("./data/processed/manufacturing_jobs_clean.csv")
df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


df.head(3)

Rows: 388
Columns: ['employer_name', 'job_title', 'salary_raw', 'salary_min', 'salary_max', 'platform', 'position_link', 'career_website', 'role_family', 'pipeline_level', 'accessibility_score']


,employer_name,job_title,salary_raw,salary_min,salary_max,platform,position_link,career_website,role_family,pipeline_level,accessibility_score
0,Boeing,Yocto Linux Software Engineer (Virtual) Senior,"$123,250.00 - $192,050.00",123250.0,192050.0,Website,https://jobs.boeing.com/job/mesa/yocto-linux-m...,Boeing Careers,Engineering,Experienced,0.1
1,Dexcom,Sr Process Development Engineer,"$91,400.00 - $152,300.00",91400.0,152300.0,Website,https://careers.dexcom.com/careers?domain=dexc...,Dexcom Careers and Job Opportunities,Engineering,Experienced,0.1
2,Dexcom,Director of Medical Device Engineering - Senso...,"$190,100.00 - $316,800.00",190100.0,316800.0,Website,https://careers.dexcom.com/careers?domain=dexc...,Dexcom Careers and Job Opportunities,Engineering,Experienced,0.1


## 2) Baseline accessible demand calculation (helper functions)

Not all job postings are realistic for undergraduates.
Here, I calculate the **total undergraduate-accessible demand**
by adding up the accessibility scores across all postings.

In [ ]:
required = ["role_family", "pipeline_level", "accessibility_score"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected modeled columns: {missing}")

# helper
def total_accessible_demand(df_in):
    return df_in["accessibility_score"].sum()

def role_family_summary(df_in):
    return (
        df_in.groupby("role_family")
             .agg(total_postings=("job_title", "count"),
                  accessible_demand=("accessibility_score", "sum"))
             .sort_values("accessible_demand", ascending=False)
             .reset_index()
    )

# baseline numbers
baseline_accessible = total_accessible_demand(df)
role_summary = role_family_summary(df)
print(f"Baseline accessible demand (placement units): {baseline_accessible:.1f}")
role_summary.head()

Baseline accessible demand (placement units): 193.9


,role_family,total_postings,accessible_demand
0,Engineering,182,95.0
1,Other,171,83.7
2,Project Management,17,7.4
3,Supply Chain,5,3.5
4,Manufacturing Operations,4,1.9


## 3) How many students can CSU realistically place?

What if accessibility for internships increased?

CSU can only place a limited number of students in jobs or internships.
Since I don’t have exact numbers, I start with a **reasonable placeholder value**.

This lets me test scenarios without claiming this is the true number.

This value is meant to be changed later if real CSU data is available.

In [ ]:
# Tunable parameter
csu_supply_default = 200.0 # for future reference, if you have actual real data, replace this value
csu_supply_default

200.0

## 4) Can CSU meet this metric or demand?

Here I compare:
- how many accessible jobs exist
- how many students CSU can place

This shows whether there is more demand than supply, or the other way around.

In [6]:
def compute_gap_metrics(df_in, csu_supply):
    total_demand = total_accessible_demand(df_in)
    placements_possible = min(total_demand, csu_supply)
    unmet_gap = max(total_demand - csu_supply, 0.0)
    percent_fulfilled = 100.0 * placements_possible / total_demand if total_demand > 0 else np.nan
    return {
        "total_accessible_demand": total_demand,
        "csu_supply": csu_supply,
        "placements_possible": placements_possible,
        "unmet_gap": unmet_gap,
        "percent_fulfilled": percent_fulfilled
    }

baseline_metrics = compute_gap_metrics(df, csu_supply_default)
baseline_metrics

{'total_accessible_demand': np.float64(193.90000000000003),
 'csu_supply': 200.0,
 'placements_possible': np.float64(193.90000000000003),
 'unmet_gap': 0.0,
 'percent_fulfilled': np.float64(100.0)}

**Interpretation:**  
- `total_accessible_demand` expresses the effective number of undergraduate-suitable placement-units
  present among all partner postings (sum of accessibility scores).
- `placements_possible` is the number of placements CSU could fill given supply.
- `percent_fulfilled` shows what share of accessible demand CSU can meet.

## 5) Scenarios and What If Questions

Now I test a few simple ideas:

1. What if CSU could place more students?
2. What if CSU focused more on its biggest partner companies?
3. What if both happened at the same time?

These are not predictions — they are just comparisons.

In [7]:
# Scenario parameters
supply_increase_pct = 0.20
target_top_k = 5
target_accessibility_delta = 0.25

# Compute top-K employers by posting count
employer_counts = df["employer_name"].value_counts().rename_axis("employer_name").reset_index(name="posting_count")
top_employers = employer_counts.head(target_top_k)["employer_name"].tolist()
top_employers

['Boeing', 'Dexcom', 'Tesla', 'ASML', 'Procter & Gamble']

## 6) Scenario implementations (functions)

In [8]:
def scenario_supply_increase(df_in, csu_supply, pct_increase):
    new_supply = csu_supply * (1 + pct_increase)
    metrics = compute_gap_metrics(df_in, new_supply)
    metrics["scenario"] = f"Supply +{int(pct_increase*100)}%"
    metrics["csu_supply_used"] = new_supply
    return metrics

def scenario_target_employers(df_in, csu_supply, employers, delta):
    df_mod = df_in.copy()
    mask = df_mod["employer_name"].isin(employers)
    # raise accessibility where mask True
    df_mod.loc[mask, "accessibility_score"] = (
        df_mod.loc[mask, "accessibility_score"] + delta
    ).clip(upper=1.0)
    metrics = compute_gap_metrics(df_mod, csu_supply)
    metrics["scenario"] = f"Target Top {len(employers)} Employers (+{delta:.2f} acc)"
    metrics["affected_postings"] = int(mask.sum())
    return metrics, df_mod

def scenario_combined(df_in, csu_supply, pct_increase, employers, delta):
    # apply employer targeting then increase supply
    metrics_target, df_mod = scenario_target_employers(df_in, csu_supply, employers, delta)
    new_supply = csu_supply * (1 + pct_increase)
    metrics = compute_gap_metrics(df_mod, new_supply)
    metrics["scenario"] = f"Combined: Supply +{int(pct_increase*100)}% & Target Top {len(employers)}"
    metrics["affected_postings"] = int(df_in["employer_name"].isin(employers).sum())
    metrics["csu_supply_used"] = new_supply
    return metrics, df_mod


## 7) Scenario Outcome Comparisons

Here I calculate what changes under each scenario and compare the results
to the current situation.


In [9]:
scenarios = []

# Baseline
scenarios.append({
    "scenario": "Baseline",
    **baseline_metrics
})

# Scenario A: supply increase
scenarios.append(scenario_supply_increase(df, csu_supply_default, supply_increase_pct))

# Scenario B: target top employers
metrics_target, df_targeted = scenario_target_employers(df, csu_supply_default, top_employers, target_accessibility_delta)
scenarios.append(metrics_target)

# Scenario C: combined
metrics_combined, df_combined = scenario_combined(df, csu_supply_default, supply_increase_pct, top_employers, target_accessibility_delta)
scenarios.append(metrics_combined)

scenarios_df = pd.DataFrame(scenarios).set_index("scenario")
scenarios_df[[
    "total_accessible_demand",
    "csu_supply",
    "csu_supply_used",
    "placements_possible",
    "unmet_gap",
    "percent_fulfilled",
    "affected_postings"
]].fillna("").T

scenario,Baseline,Supply +20%,Target Top 5 Employers (+0.25 acc),Combined: Supply +20% & Target Top 5
total_accessible_demand,193.9,193.9,249.15,249.15
csu_supply,200.0,240.0,200.0,240.0
csu_supply_used,,240.0,,240.0
placements_possible,193.9,193.9,200.0,240.0
unmet_gap,0.0,0.0,49.15,9.15
percent_fulfilled,100.0,100.0,80.272928,96.327514
affected_postings,,,305.0,305.0


## 8) How much better is each option?

This section looks at how many **additional student placements**
each scenario could produce compared to doing nothing.

In [10]:
# Baseline placements
baseline_placements = baseline_metrics["placements_possible"]

def placements_from_metrics(m):
    return m["placements_possible"]

impact_rows = []
for s in scenarios:
    name = s["scenario"]
    placements = placements_from_metrics(s)
    delta = placements - baseline_placements
    impact_rows.append({
        "scenario": name,
        "placements": placements,
        "delta_vs_baseline": delta,
        "percent_fulfilled": s.get("percent_fulfilled", np.nan),
        "affected_postings": s.get("affected_postings", 0)
    })

impact_df = pd.DataFrame(impact_rows).set_index("scenario")
impact_df

,placements,delta_vs_baseline,percent_fulfilled,affected_postings
scenario,,,,
Baseline,193.9,0.0,100.000000,0
Supply +20%,193.9,0.0,100.000000,0
Target Top 5 Employers (+0.25 acc),200.0,6.1,80.272928,305
Combined: Supply +20% & Target Top 5,240.0,46.1,96.327514,305


## 9) Where do the gains come from?

Here I check which types of roles benefit the most under the combined scenario.
This helps explain *why* a scenario works.

In [11]:
# role-family summary for baseline vs combined
role_baseline = role_family_summary(df).set_index("role_family")
role_combined = role_family_summary(df_combined).set_index("role_family")

role_compare = role_baseline.join(role_combined, lsuffix="_baseline", rsuffix="_combined", how="outer").fillna(0)
role_compare["additional_accessible_demand"] = role_compare["accessible_demand_combined"] - role_compare["accessible_demand_baseline"]
role_compare = role_compare.sort_values("accessible_demand_baseline", ascending=False)
role_compare[[
    "total_postings_baseline", "accessible_demand_baseline", 
    "total_postings_combined", "accessible_demand_combined",
    "additional_accessible_demand"
]]


,total_postings_baseline,accessible_demand_baseline,total_postings_combined,accessible_demand_combined,additional_accessible_demand
role_family,,,,,
Engineering,182,95.0,182,122.00,27.00
Other,171,83.7,171,105.45,21.75
Project Management,17,7.4,17,10.65,3.25
Supply Chain,5,3.5,5,4.00,0.50
Manufacturing Operations,4,1.9,4,2.65,0.75
Quality & Compliance,6,1.8,6,3.30,1.50
Design,3,0.6,3,1.10,0.50


## 10) Results summary & interpretation: What does this tell us?

This section summarizes what the numbers above are saying
in plain language.

In [12]:
from textwrap import dedent

print(dedent(f"""
BASELINE:
- Accessible demand (placement units): {baseline_metrics['total_accessible_demand']:.1f}
- CSU supply (default param): {baseline_metrics['csu_supply']:.1f}
- Placements possible: {baseline_metrics['placements_possible']:.1f}
- Percent of accessible demand CSU can meet: {baseline_metrics['percent_fulfilled']:.1f}%

SCENARIO COMPARISON (deltas vs baseline):
{impact_df.to_string()}
"""))



BASELINE:
- Accessible demand (placement units): 193.9
- CSU supply (default param): 200.0
- Placements possible: 193.9
- Percent of accessible demand CSU can meet: 100.0%

SCENARIO COMPARISON (deltas vs baseline):
                                      placements  delta_vs_baseline  percent_fulfilled  affected_postings
scenario                                                                                                 
Baseline                                   193.9                0.0         100.000000                  0
Supply +20%                                193.9                0.0         100.000000                  0
Target Top 5 Employers (+0.25 acc)         200.0                6.1          80.272928                305
Combined: Supply +20% & Target Top 5       240.0               46.1          96.327514                305



### Key decision insights (how to read these numbers)
- If the **Supply +20%** scenario produces the largest delta in placements, this suggests investing in more internship slots / program capacity is efficient.
- If the **Target Top Employers** scenario produces a large delta despite affecting relatively few postings, this suggests employer-targeted interventions (partnership programs to convert Website postings into student-facing roles) are high-leverage.
- The **Combined** scenario indicates synergy: raising accessibility at top employers *and* increasing CSU capacity produces the most placements.

## 11) Simple cost-effectiveness heuristic (optional)

If you have cost estimates (e.g., cost-per-placement to expand internships, or cost to run employer engagement), you can compute `cost_per_additional_placement` for each scenario.

Example (illustrative):
- cost_per_new_intern = $4,000 (program stipend + admin)
- cost_of_employer_engagement = $10,000 (one-time program) to convert N postings

This notebook does not include costs by default, but the structure is ready to accept them for ROI calculations.


## 12) What this analysis does and does not do

This analysis is meant to explore ideas, not make exact predictions.
It depends on assumptions and simplified scoring.

## Simple takeaway

Based on these comparisons:
- increasing student placement capacity helps
- focusing on the largest partner companies helps
- doing both works best

These results suggest CSU should focus on targeted, high-impact efforts
rather than spreading resources evenly.